# Notebook 01 — Data Preparation
Downloads the SpamAssassin Public Corpus, parses raw email files, cleans text, and exports `data/emails.csv`.

## Cell 1 — Imports & configuration

This cell initialises the notebook environment by importing all required libraries
and defining the directory structure as named constants.

Centralising paths here ensures that if the project is moved or restructured,
only this cell needs to be updated — all other cells reference these constants
rather than hardcoded strings.

In [2]:
# Cell 1 — Imports & Config
import os
import re
import pandas as pd
from html.parser import HTMLParser

# ── Constants ──────────────────────────────────────────────
DATA_DIR   = os.path.join('..', 'data')
RAW_DIR    = os.path.join(DATA_DIR, 'raw')
SPAM_DIR   = os.path.join(RAW_DIR, 'spam')
HAM_DIR    = os.path.join(RAW_DIR, 'easy_ham')
OUTPUT_CSV = os.path.join(DATA_DIR, 'emails.csv')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_DIR,  exist_ok=True)

print('Directories ready.')
print(f'  SPAM_DIR   : {SPAM_DIR}')
print(f'  HAM_DIR    : {HAM_DIR}')
print(f'  OUTPUT_CSV : {OUTPUT_CSV}')

Directories ready.
  SPAM_DIR   : ..\data\raw\spam
  HAM_DIR    : ..\data\raw\easy_ham
  OUTPUT_CSV : ..\data\emails.csv


## Cell 2 — Download Dataset

This cell downloads the SpamAssassin Public Corpus directly from the Apache
foundation's public archive — a widely used benchmark dataset in email
security research.

Two corpora are retrieved: a spam collection and a legitimate email collection.
Both are extracted automatically into `data/raw/`.

| Label | URL |
|-------|-----|
| Spam (label=1) | https://spamassassin.apache.org/old/publiccorpus/20050311_spam_2.tar.bz2 |
| Ham  (label=0) | https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2 |


In [3]:
# Cell 2 — Download Dataset (auto-download via Python)
import urllib.request
import tarfile

DOWNLOADS = [
    (
        'https://spamassassin.apache.org/old/publiccorpus/20050311_spam_2.tar.bz2',
        os.path.join(RAW_DIR, 'spam.tar.bz2')
    ),
    (
        'https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2',
        os.path.join(RAW_DIR, 'ham.tar.bz2')
    ),
]

for url, dest in DOWNLOADS:
    if not os.path.exists(dest):
        print(f'Downloading {url} ...')
        urllib.request.urlretrieve(url, dest)
        print(f'  Saved → {dest}')
    else:
        print(f'  Already downloaded: {dest}')

# Extract
for _, tarpath in DOWNLOADS:
    print(f'Extracting {tarpath} ...')
    with tarfile.open(tarpath, 'r:bz2') as tf:
        tf.extractall(RAW_DIR)
    print('  Done.')

# Rename extracted folders to match our constants if needed
for name, expected in [('spam_2', 'spam'), ('easy_ham', 'easy_ham')]:
    src_path = os.path.join(RAW_DIR, name)
    dst_path = os.path.join(RAW_DIR, expected)
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        os.rename(src_path, dst_path)

print('\nFolder check:')
for d in [SPAM_DIR, HAM_DIR]:
    count = len([f for f in os.listdir(d) if not f.startswith('cmds')]) if os.path.exists(d) else 0
    print(f'  {d}  → {count} files')

  Saved → ..\data\raw\spam.tar.bz2
  Saved → ..\data\raw\ham.tar.bz2
Extracting ..\data\raw\spam.tar.bz2 ...
  Done.
Extracting ..\data\raw\ham.tar.bz2 ...
  Done.

Folder check:
  ..\data\raw\spam  → 1396 files
  ..\data\raw\easy_ham  → 2500 files


## Cell 3 — Email parsing & text cleaning

Raw email files contain significant noise that is irrelevant to classification:
routing headers, MIME metadata, HTML markup, and encoding artefacts. This cell
defines a parsing function that strips all of it away and retains only the
content a human would actually read.

The parsing pipeline applies three steps in sequence:

1. **Header removal** — all lines before the blank separator are discarded,
   with the exception of the Subject field which is preserved as a signal.
2. **HTML stripping** — markup tags are removed using a dedicated parser,
   leaving only visible text.
3. **Whitespace normalisation** — all whitespace sequences are collapsed to a
   single space, producing clean, uniform strings.

Files that yield fewer than 20 characters after cleaning are discarded,
as they carry no meaningful signal.

In [4]:
# Cell 3 — Parse Email Files

class _HTMLStripper(HTMLParser):
    """Strips HTML tags and collects visible text."""
    def __init__(self):
        super().__init__()
        self._parts = []
    def handle_data(self, data):
        self._parts.append(data)
    def get_text(self):
        return ' '.join(self._parts)

def _strip_html(text: str) -> str:
    """Remove HTML tags from text."""
    stripper = _HTMLStripper()
    try:
        stripper.feed(text)
        return stripper.get_text()
    except Exception:
        # Fallback: regex-based strip
        return re.sub(r'<[^>]+>', ' ', text)

def parse_email(filepath: str) -> str:
    """
    Read a raw SpamAssassin email file.
    - Skip all headers EXCEPT the Subject line.
    - Keep the body (everything after the blank separator line).
    - Strip HTML tags.
    - Collapse whitespace.
    Returns a single clean string, or '' on failure.
    """
    try:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as fh:
            raw = fh.read()
    except Exception:
        return ''

    subject = ''
    body    = ''
    in_headers = True
    lines = raw.splitlines()

    for i, line in enumerate(lines):
        if in_headers:
            if line.lower().startswith('subject:'):
                subject = line[8:].strip()          # capture subject text
            if line.strip() == '':                  # blank line = end of headers
                body = '\n'.join(lines[i+1:])
                in_headers = False
                break

    combined = f"{subject} {body}"
    combined = _strip_html(combined)
    combined = re.sub(r'\s+', ' ', combined).strip()
    return combined


# ── Loop through spam and ham folders ──────────────────────
records = []

for folder, label in [(SPAM_DIR, 1), (HAM_DIR, 0)]:
    files = [
        f for f in os.listdir(folder)
        if not f.startswith('cmds') and not f.startswith('.')
    ]
    print(f'Parsing {len(files)} files from {folder} (label={label}) ...')
    for fname in files:
        fpath = os.path.join(folder, fname)
        text  = parse_email(fpath)
        if len(text) > 20:          # skip near-empty parses
            records.append({'text': text, 'label': label})

print(f'\nTotal records parsed: {len(records)}')

Parsing 1396 files from ..\data\raw\spam (label=1) ...
Parsing 2500 files from ..\data\raw\easy_ham (label=0) ...

Total records parsed: 3882


## Cell 4 — DataFrame construction & class balance check

The parsed records are collected into a pandas DataFrame with two columns:
`text` (the cleaned email content) and `label` (1 for spam, 0 for legitimate).

A class distribution report is printed to verify that the dataset meets the
minimum requirements for reliable model training — at least 2,000 records with
a reasonably balanced split between classes. An assertion enforces this
threshold so any downstream notebook fails fast rather than silently training
on insufficient data.

A random sample of three rows is displayed for a quick visual quality check.

In [5]:
# Cell 4 — Build DataFrame

df = pd.DataFrame(records)          # columns: text, label

print('Shape          :', df.shape)
print('\nLabel distribution:')
print(df['label'].value_counts())
print(f'  Spam (1): {(df.label==1).sum()}  |  Ham (0): {(df.label==0).sum()}')

# Balance check
total = len(df)
spam_pct = (df.label==1).sum() / total * 100
ham_pct  = (df.label==0).sum() / total * 100
print(f'\nBalance: {spam_pct:.1f}% spam / {ham_pct:.1f}% ham')
assert total >= 2000, f'Expected ≥2000 rows, got {total}'
print(f'\n✓ Row count check passed ({total} rows)')

print('\nSample rows:')
df.sample(3, random_state=42)[['label', 'text']].assign(
    text=df['text'].str[:120] + '...'
)

Shape          : (3882, 2)

Label distribution:
label
0    2500
1    1382
Name: count, dtype: int64
  Spam (1): 1382  |  Ham (0): 2500

Balance: 35.6% spam / 64.4% ham

✓ Row count check passed (3882 rows)

Sample rows:


,label,text
3432,0,Mac OS X browsers URL: http://www.askbjoernhan...
1018,1,FREE e-Book Publishing Software - Download NOW...
1941,0,Keillor voices up on what to do with the WTC s...


## Cell 5 — Persist & validate output

The cleaned DataFrame is written to `data/emails.csv` without a row index.
The file is immediately reloaded from disk and subjected to three validation
assertions before the notebook is considered complete.

| Check |
|-------|
| Column `text` present |
| Column `label` present |
| Row count matches in-memory DataFrame |



In [6]:
# Cell 5 — Save & Validate

df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved → {OUTPUT_CSV}')

# Reload and validate
df_check = pd.read_csv(OUTPUT_CSV)
assert 'text'  in df_check.columns, 'Missing column: text'
assert 'label' in df_check.columns, 'Missing column: label'
assert len(df_check) == len(df),    'Row count mismatch after save'

print(f'\n✓ All assertions passed.')
print(f'  Shape  : {df_check.shape}')
print(f'  Columns: {list(df_check.columns)}')
print(f'  Labels : {df_check.label.value_counts().to_dict()}')
print('\nFirst 3 rows:')
display(df_check.head(3))

Saved → ..\data\emails.csv

✓ All assertions passed.
  Shape  : (3882, 2)
  Columns: ['text', 'label']
  Labels : {0: 2500, 1: 1382}

First 3 rows:


,text,label
0,[ILUG] STOP THE MLM INSANITY Greetings! You ar...,1
1,"Real Protection, Stun Guns! Free Shipping! Tim...",1
2,"New Improved Fat Burners, Now With TV Fat Abso...",1
